#### Lecture de la bd

In [37]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler

In [38]:
pd.set_option('display.max_columns', None)

df = pd.read_csv("./echantillon.csv")
df.head()

,Id,OrgId,IncidentId,AlertId,Timestamp,DetectorId,AlertTitle,Category,MitreTechniques,IncidentGrade,ActionGrouped,ActionGranular,EntityType,EvidenceRole,DeviceId,Sha256,IpAddress,Url,AccountSid,AccountUpn,AccountObjectId,AccountName,DeviceName,NetworkMessageId,EmailClusterId,RegistryKey,RegistryValueName,RegistryValueData,ApplicationId,ApplicationName,OAuthApplicationId,ThreatFamily,FileName,FolderPath,ResourceIdName,ResourceType,Roles,OSFamily,OSVersion,AntispamDirection,SuspicionLevel,LastVerdict,CountryCode,State,City
0,884763264609,21,111280,263403,2024-06-09T11:29:01.000Z,3,4,SuspiciousActivity,NaN,BenignPositive,NaN,NaN,Machine,Impacted,1214,138268,360606,160396,441377,673934,425863,453297,2759,529644,NaN,1631,635,860,2251,3421,881,NaN,289573,117668,3586,NaN,NaN,5,66,NaN,Suspicious,Suspicious,242,1445,10630
1,515396079824,36,30247,97984,2024-06-05T09:54:14.000Z,10,8,InitialAccess,T1566.001,BenignPositive,NaN,NaN,Mailbox,Impacted,98799,138268,360606,160396,68110,80214,56600,61887,153085,529644,NaN,1631,635,860,2251,3421,881,NaN,289573,117668,3586,NaN,NaN,5,66,NaN,NaN,NoThreatsFound,242,1445,10630
2,1511828491965,12,2420,1652,2024-05-23T01:55:44.000Z,16,186,Impact,NaN,BenignPositive,NaN,NaN,Ip,Related,98799,138268,894,160396,441377,673934,425863,453297,153085,529644,NaN,1631,635,860,2251,3421,881,NaN,289573,117668,3586,NaN,NaN,5,66,NaN,NaN,NaN,242,1445,10630
3,223338304201,38,289307,693576,2024-06-13T12:58:35.000Z,2,2,CommandAndControl,NaN,BenignPositive,NaN,NaN,Machine,Impacted,70591,138268,360606,160396,441377,673934,425863,453297,99846,529644,NaN,1631,635,860,2251,3421,881,NaN,289573,117668,3586,NaN,NaN,0,0,NaN,Suspicious,Suspicious,242,1445,10630
4,335007450734,26,60218,421825,2024-06-10T07:23:12.000Z,497,539,Discovery,T1069.001;T1069.002,BenignPositive,NaN,NaN,Process,Related,98799,27,360606,160396,441377,673934,425863,453297,153085,529644,NaN,1631,635,860,2251,3421,881,NaN,5,0,3586,NaN,NaN,5,66,NaN,NaN,NaN,242,1445,10630


## 1. Préparation des données

### Traitement des valeurs manquantes
On va d'abord vérifier le pourcentage de valeurs manquantes par colonne, puis on supprimera les colonnes qui ont trop de valeurs manquantes (> 30%). 
Pour les autres, on va appliquer une stratégie d'imputation avec la médiane et le mode


In [39]:
# Calcul du pourcentage de valeurs manquantes
missing_percentages = df.isnull().sum() / len(df) * 100
print("Pourcentage de valeurs manquantes :\n", missing_percentages[missing_percentages > 0].sort_values(ascending=False))

Pourcentage de valeurs manquantes :
 ActionGrouped        99.954564
ActionGranular       99.954564
ResourceType         99.932374
ThreatFamily         99.234980
EmailClusterId       98.963418
AntispamDirection    98.105412
Roles                97.586593
SuspicionLevel       84.814768
LastVerdict          76.392147
MitreTechniques      57.127158
dtype: float64


In [40]:
# Suppression des colonnes avec beaucoup de valeurs manquantes
cols_to_drop = missing_percentages[missing_percentages > 30].index
df_clean = df.drop(columns=cols_to_drop)

# Variables quantitatives : Remplacement par la médiane 
col_numeriques = df_clean.select_dtypes(include=['int64', 'float64']).columns
for col in col_numeriques:
    if df_clean[col].isnull().sum() > 0:
        df_clean[col] = df_clean[col].fillna(df_clean[col].median())

# Variables qualitatives : Remplacement par le mode
col_categorielles = df_clean.select_dtypes(include=['object', 'category']).columns
for col in col_categorielles:
    if df_clean[col].isnull().sum() > 0:
        df_clean[col] = df_clean[col].fillna(df_clean[col].mode()[0])

print(f"Dataset original: {df.shape[0]} lignes, {df.shape[1]} colonnes")
print(f"Dataset nettoyé: {df_clean.shape[0]} lignes, {df_clean.shape[1]} colonnes")
print(f"Valeurs manquantes restantes au total : {df_clean.isnull().sum().sum()}")


Dataset original: 94638 lignes, 45 colonnes
Dataset nettoyé: 94638 lignes, 35 colonnes
Valeurs manquantes restantes au total : 0


/var/folders/43/w_4sk0b97cv9bp65xqk5yg3m0000gn/T/ipykernel_41831/174883985.py:12: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  col_categorielles = df_clean.select_dtypes(include=['object', 'category']).columns


### Normalisation des données
On va standardiser les colonnes numériques (moyenne=0, ecart-type=1) à l'aide de StandardScaler de scikit-learn. Cette étape aide les algorithmes d'apprentissage à mieux performer sans donner plus de poids aux attributs ayant de grandes valeurs.
**Attention** : On ne doit évidemment pas normaliser les colonnes qui servent d'identifiants (ID) ou la variable cible (si elle est numérique) !


In [41]:
from sklearn.preprocessing import StandardScaler

# On sélectionne toutes les colonnes numériques
all_numeric_cols = df_clean.select_dtypes(include=['int64', 'float64']).columns

cols_to_exclude = [col for col in all_numeric_cols if 'Id' in col or 'id' in col.lower()]

variable_cible = 'IncidentGrade' 
if variable_cible in all_numeric_cols and variable_cible not in cols_to_exclude:
    cols_to_exclude.append(variable_cible)

# Liste finale des colonnes numériques à normaliser
numeric_cols_to_scale = [col for col in all_numeric_cols if col not in cols_to_exclude]

print(f"Colonnes exclues de la normalisation : {cols_to_exclude}")

scaler = StandardScaler()

# Application de la standardisation
if len(numeric_cols_to_scale) > 0:
    df_clean[numeric_cols_to_scale] = scaler.fit_transform(df_clean[numeric_cols_to_scale])

df_clean[numeric_cols_to_scale].head()


Colonnes exclues de la normalisation : ['Id', 'OrgId', 'IncidentId', 'AlertId', 'DetectorId', 'DeviceId', 'AccountSid', 'AccountObjectId', 'NetworkMessageId', 'ApplicationId', 'OAuthApplicationId', 'ResourceIdName']


,AlertTitle,Sha256,IpAddress,Url,AccountUpn,AccountName,DeviceName,RegistryKey,RegistryValueName,RegistryValueData,ApplicationName,FileName,FolderPath,OSFamily,OSVersion,CountryCode,State,City
0,-0.254497,0.281868,0.533274,0.268338,0.716272,0.547829,-3.872913,0.041075,0.020606,0.02252,0.1571,0.333943,0.309889,0.143659,0.143793,0.29238,0.266409,0.266265
1,-0.254135,0.281868,0.533274,0.268338,-1.336770,-1.707494,0.274717,0.041075,0.020606,0.02252,0.1571,0.333943,0.309889,0.143659,0.143793,0.29238,0.266409,0.266265
2,-0.238048,0.281868,-2.006578,0.268338,0.716272,0.547829,0.274717,0.041075,0.020606,0.02252,0.1571,0.333943,0.309889,0.143659,0.143793,0.29238,0.266409,0.266265
3,-0.254678,0.281868,0.533274,0.268338,0.716272,0.547829,-1.194195,0.041075,0.020606,0.02252,0.1571,0.333943,0.309889,-6.995402,-6.968561,0.29238,0.266409,0.266265
4,-0.206143,-3.773484,0.533274,0.268338,0.716272,0.547829,0.274717,0.041075,0.020606,0.02252,0.1571,-3.231235,-3.361475,0.143659,0.143793,0.29238,0.266409,0.266265
